In [1]:
!pip install --upgrade "pandas>=2.3.0"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 20.6 MB/s  0:00:00
  Attempting uninstall: pandas
    Found existing installation: pandas 2.2.3
    Uninstalling pandas-2.2.3:
      Successfully uninstalled pandas-2.2.3
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires pandas==2.2.3, but you have pandas 3.1.1 which is incompatible.


In [1]:
from google.colab import files
import pandas as pd
import joblib
import shutil

print("Active pandas version:", pd.__version__)

# Click "Choose Files" when prompted and select apriori_recommendation_rules.joblib
uploaded = files.upload()

# Normalize filename in case browser appends (1)
uploaded_name = next(iter(uploaded))
if uploaded_name != "apriori_recommendation_rules.joblib":
    shutil.move(uploaded_name, "apriori_recommendation_rules.joblib")

artifact = joblib.load("apriori_recommendation_rules.joblib")
print("Success! Loaded keys:", list(artifact.keys()))

/usr/local/lib/python3.13/dist-packages/pandas/core/arrays/masked.py:63: UserWarning: Pandas requires version '1.5.0' or newer of 'bottleneck' (version '1.4.2' currently installed).
  from pandas.core import (


Active pandas version: 3.1.1


Saving apriori_recommendation_rules.joblib to apriori_recommendation_rules.joblib
Success! Loaded keys: ['cleaned_rules', 'recommendation_lookup', 'sku_lookup']


In [2]:
%%writefile apriori_endpoint.py
import joblib
from flask import Flask, jsonify, request

app = Flask(__name__)

# Load the saved Apriori artifact dictionary
ARTIFACT_PATH = "apriori_recommendation_rules.joblib"
artifact = joblib.load(ARTIFACT_PATH)

rules_df = artifact["cleaned_rules"]
sku_lookup = artifact["sku_lookup"]

# Reverse lookup (lowercase product_name -> sku) so inputs can be SKUs or product names
name_to_sku = {
    row["product_name"].strip().lower(): sku
    for sku, row in sku_lookup.iterrows()
}


def normalize_basket_to_skus(basket):
    """Converts a list of SKUs or product names into a set of valid SKU codes."""
    sku_set = set()
    for item in basket:
        clean_item = str(item).strip()
        upper_item = clean_item.upper()
        lower_item = clean_item.lower()

        if upper_item in sku_lookup.index:
            sku_set.add(upper_item)
        elif lower_item in name_to_sku:
            sku_set.add(name_to_sku[lower_item])
    return sku_set


def enrich_sku_details(sku):
    """Fetches product metadata (name, category, KES price) for a given SKU."""
    if sku in sku_lookup.index:
        row = sku_lookup.loc[sku]
        return {
            "sku": sku,
            "product_name": row["product_name"],
            "category": row["category"],
            "unit_price_kes": float(row["unit_price_kes"]),
        }
    return {"sku": sku, "product_name": "Unknown", "category": "Unknown", "unit_price_kes": 0.0}


def get_recommendations(basket, top_n=5):
    basket_skus = normalize_basket_to_skus(basket)
    if not basket_skus:
        return []

    # Filter rules where all antecedent SKUs are inside the user's basket
    matching_rules = rules_df[
        rules_df["antecedents"].apply(lambda ant: ant.issubset(basket_skus))
    ]

    if matching_rules.empty:
        return []

    # Rank strongest rules by lift, then confidence
    matching_rules = matching_rules.sort_values(
        by=["lift", "confidence"], ascending=False
    )

    recommendations = []
    seen_skus = set()

    for _, row in matching_rules.iterrows():
        for rec_sku in row["consequents"]:
            if rec_sku not in basket_skus and rec_sku not in seen_skus:
                seen_skus.add(rec_sku)
                details = enrich_sku_details(rec_sku)
                recommendations.append({
                    **details,
                    "based_on_skus": sorted(list(row["antecedents"])),
                    "confidence": round(float(row["confidence"]), 4),
                    "lift": round(float(row["lift"]), 4),
                    "support": round(float(row["support"]), 4),
                })
                if len(recommendations) >= top_n:
                    return recommendations

    return recommendations


@app.route("/recommend", methods=["POST"])
def recommend():
    data = request.get_json(silent=True)

    if not data or "basket" not in data:
        return jsonify({
            "error": "Invalid request. Provide a JSON body with a 'basket' list."
        }), 400

    basket = data["basket"]
    if not isinstance(basket, list) or len(basket) == 0:
        return jsonify({
            "error": "'basket' must be a non-empty list of SKUs or product names."
        }), 400

    top_n = data.get("top_n", 5)

    try:
        recs = get_recommendations(basket, top_n=int(top_n))
        return jsonify({
            "input_basket": basket,
            "matched_skus": sorted(list(normalize_basket_to_skus(basket))),
            "recommendations": recs,
            "count": len(recs),
        }), 200
    except Exception as e:
        return jsonify({"error": str(e)}), 500


if __name__ == "__main__":
    app.run(host="0.0.0.0", port=5004)

Writing apriori_endpoint.py


In [3]:
import sys
import json
sys.path.append("/content")

from apriori_endpoint import app

client = app.test_client()

# Test 1: Using SKU codes directly
res_sku = client.post("/recommend", json={"basket": ["P03", "P04"], "top_n": 3})
print("=== Test 1 (SKU Input) ===")
print("Status:", res_sku.status_code)
print(json.dumps(res_sku.get_json(), indent=2))

# Test 2: Using human-readable product names
res_name = client.post("/recommend", json={"basket": ["Wireless Mouse", "O04"], "top_n": 3})
print("\n=== Test 2 (Product Name Input) ===")
print("Status:", res_name.status_code)
print(json.dumps(res_name.get_json(), indent=2))

=== Test 1 (SKU Input) ===
Status: 200
{
  "count": 2,
  "input_basket": [
    "P03",
    "P04"
  ],
  "matched_skus": [
    "P03",
    "P04"
  ],
  "recommendations": [
    {
      "based_on_skus": [
        "P03",
        "P04"
      ],
      "category": "Photography",
      "confidence": 0.9367,
      "lift": 19.1165,
      "product_name": "Digital Camera",
      "sku": "P01",
      "support": 0.0148,
      "unit_price_kes": 45000.0
    },
    {
      "based_on_skus": [
        "P03",
        "P04"
      ],
      "category": "Photography",
      "confidence": 0.9367,
      "lift": 19.1165,
      "product_name": "Memory Card 128GB",
      "sku": "P02",
      "support": 0.0148,
      "unit_price_kes": 2200.0
    }
  ]
}

=== Test 2 (Product Name Input) ===
Status: 200
{
  "count": 2,
  "input_basket": [
    "Wireless Mouse",
    "O04"
  ],
  "matched_skus": [
    "O02",
    "O04"
  ],
  "recommendations": [
    {
      "based_on_skus": [
        "O02",
        "O04"
      ],
      "ca